# Train YOLOv8n trên Kaggle

Khác với bản Colab:
- Thư mục chính là **/kaggle/working** (không phải /content)
- **Không cần mount Drive** — lấy kết quả bằng nút *Output → Download* bên phải
- Kaggle miễn phí có **30 giờ GPU/tuần** và 1 session chạy liên tục tới ~9-12h → ít bị ngắt giữa chừng hơn Colab

**Bước đầu tiên:** mở panel **Settings ⚙** bên phải → *Accelerator*: chọn **GPU T4 x2** (hoặc P100) → bật *Internet* ON (để pip + tải dataset) → *Save*.

In [ ]:
!nvidia-smi


In [ ]:
!pip install -q ultralytics roboflow

from ultralytics import YOLO
import ultralytics, torch
print("Ultralytics:", ultralytics.__version__)
print("GPU available:", torch.cuda.is_available())

## 1. Tải dataset từ Roboflow

Dataset được export sang YOLOv8, kèm data.yaml (chứa tên class). Cần *Internet* bật trong Settings.

In [ ]:
# Tải dataset từ Roboflow — KHÔNG hardcode key (tránh lộ trên GitHub).
# Key đọc theo thứ tự: Kaggle Secrets -> env var -> file .env
#
# Trên Kaggle: nút Add-ons / Settings -> Secrets -> tạo "ROBOFLOW_API_KEY"
# rồi dán key vào. Hoặc: mở Code -> ổ khóa "Add secret" bên phải.
# (Kaggle không đọc được file .env — phải dùng Secrets.)

def _get_roboflow_key():
    import os
    # 1) Kaggle Secrets (chạy trên Kaggle)
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
        if key:
            return key
    except Exception:
        pass
    # 2) Biến môi trường
    if os.environ.get("ROBOFLOW_API_KEY"):
        return os.environ["ROBOFLOW_API_KEY"]
    # 3) File .env (chạy local)
    try:
        for line in open(".env"):
            line = line.strip()
            if line.startswith("ROBOFLOW_API_KEY") and "=" in line:
                return line.split("=", 1)[1].strip()
    except FileNotFoundError:
        pass
    raise ValueError(
        "Thiếu ROBOFLOW_API_KEY. Trên Kaggle: mở nút 'Add secret' (ổ khóa) "
        "bên phải, tạo secret tên ROBOFLOW_API_KEY. Chạy local thì tạo file .env."
    )

ROBOFLOW_API_KEY = _get_roboflow_key()
print("Đã lấy ROBOFLOW_API_KEY")

from roboflow import Roboflow

WORKSPACE       = "trantungbach26-gmail-com"
PROJECT_NAME    = "citrus-disease-detection-yoydc-ahtka"
PROJECT_VERSION = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT_NAME)
project.version(PROJECT_VERSION).download("yolov8")
print("Dataset đã tải về /kaggle/working/")

In [ ]:
# Tự tìm thư mục dataset vừa tải, và đổi đường dẫn sang thư mục chuẩn tổ chức
import os, glob, shutil, yaml

candidates = glob.glob("/kaggle/working/*/data.yaml")
if candidates:
    DATASET_PATH = os.path.dirname(candidates[0])
else:
    DATASET_PATH = "/kaggle/working/citrus-disease-detection-1"

print("DATASET_PATH =", DATASET_PATH)

with open(os.path.join(DATASET_PATH, "data.yaml")) as f:
    cfg = yaml.safe_load(f)
print("Số class:", cfg["nc"])
print("Tên class:", cfg["names"])

## 2. Load model & train

tự tải yolov8n.pt lần đầu (cần Internet).

In [ ]:
model = YOLO("yolov8n.pt")  # tự tải pretrained
print("Đã load yolov8n.pt")

In [ ]:
# ===== Cấu hình train (best practice) =====
# epochs cao + patience tự dừng: yolov8n trên 16k ảnh cần ~100+ epochs mới hội tụ.
# Bằng chứng: run 100 ep có đỉnh mAP50 ở epoch 57 và mAP50-95 ở 70.
#   -> 50 ep là thiếu (v1 mAP50 0.531 vs v2 0.554).
#   -> time=8 giới hạn cứng: tránh vượt session Kaggle 9-12h.
EPOCHS = 150      # mục tiêu; patience sẽ dừng sớm khi không còn cải thiện
IMGSZ = 640       # khớp với export & kmodel
BATCH = 16
PATIENCE = 15

RESULTS_DIR = "/kaggle/working/runs/drone_yolov8n/weights"
OUT_DIR     = "/kaggle/working/drone_yolo"
os.makedirs(OUT_DIR, exist_ok=True)

# ===== BACKUP mỗi epoch: copy best.pt vào thư mục Output để tải được bất kỳ lúc nào =====
from ultralytics.utils import callbacks

def _backup(trainer):
    try:
        src = os.path.join(trainer.save_dir, "weights", "best.pt")
        shutil.copy(src, os.path.join(OUT_DIR, "best_checkpoint.pt"))
        print(f"  [backup epoch {trainer.epoch}] -> {OUT_DIR}", flush=True)
    except Exception as e:
        print("  [backup fail]", e, flush=True)

callbacks.default_callbacks["on_fit_epoch_end"].append(_backup)

# ===== TĂNG RECALL (model citrus dễ bỏ sót bệnh: P cao / R thấp) =====
# - class_weights=True: lớp ít ảnh nặng hơn -> bắt được nhiều đốm bệnh hiếm
# - fliplr=0.5 + scale=0.5: đa dạng vị trí/kích thước lá
# KHÔNG dùng flipud/degrees/shear/perspective mạnh vì ảnh drone trên cao
# và lá bệnh có hình dạng có ý nghĩa — biến dạng quá làm sai nhãn.
print(f"Train yolov8n: epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, patience={PATIENCE}")

results = model.train(
    data=f"{DATASET_PATH}/data.yaml",
    epochs=EPOCHS,
    time=8,               # tối đa 8 giờ — an toàn với giới hạn session Kaggle
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    device=0,
    seed=42,
    class_weights=True,   # cân bằng lớp hiếm -> tăng recall
    fliplr=0.5,           # lật ngang
    scale=0.5,            # zoom ngẫu nhiên -> lá to nhỏ khác nhau
    cache=True,
    workers=2,
    cos_lr=True,          # giảm learning rate theo cos, hội tụ tốt hơn
    project="/kaggle/working/runs",
    name="drone_yolov8n",
)

## 3. Đánh giá kết quả

In [ ]:
# Đánh giá — in cả recall (quan trọng: model hay bỏ sót bệnh)
metrics = model.val()
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")

In [ ]:
import glob

test_imgs = (glob.glob(f"{DATASET_PATH}/test/images/*.*")
             or glob.glob(f"{DATASET_PATH}/valid/images/*.*")
             or glob.glob(f"{DATASET_PATH}/val/images/*.*"))
print("Tìm thấy", len(test_imgs), "ảnh")

if test_imgs:
    model.predict(source=test_imgs[:4], conf=0.25, save=True, project="/kaggle/working/predict")
    saved = glob.glob("/kaggle/working/predict/**/*.jpg", recursive=True)
    if saved:
        from IPython.display import Image
        print("Ảnh đã lưu:", saved[0])
        display(Image(filename=saved[0]))
    else:
        print("Không tìm thấy ảnh đã lưu (bỏ qua preview, không crash)")
else:
    print("Không có ảnh test để preview")

## 4. Export 2 phiên bản

- **best.pt** → chạy realtime trên **laptop** (realtime_cam.py)
- **best.onnx** → qua nncase → **best.kmodel** chạy trên **drone K230**

In [ ]:
# Export ONNX dành cho K230. (best.pt cho laptop đã có sẵn trong thư mục weights.)
best = YOLO(f"{RESULTS_DIR}/best.pt")
best.export(format="onnx", imgsz=IMGSZ, opset=11, simplify=True)

# Liệt kê các phiên bản đã có
print("\nĐã tạo/xuất các phiên bản:")
for f in sorted(os.listdir(RESULTS_DIR)):
    if f in ("best.pt", "best.onnx"):
        size = os.path.getsize(os.path.join(RESULTS_DIR, f)) / 1e6
        print(f"  - {f}  ({size:.1f} MB)")
print("\nThư mục:", RESULTS_DIR)

In [ ]:
# Sao chép mọi phiên bản vào thư mục Output để tải về máy
import glob as _g

for f in _g.glob(RESULTS_DIR + "/best.*"):
    shutil.copy(f, os.path.join(OUT_DIR, os.path.basename(f)))
    print("Đã copy output:", os.path.basename(f))

print("\n>>> Tải kết quả: panel bên phải tab 'Output' -> biểu tượng tải xuống (Download all).")

## 5. Tùy chọn: convert ONNX → kmodel ngay trên Kaggle (Linux)

### Tạm dừng — xem side-note:

Để chắc chắn dùng đúng bản nncase của tài liệu CanMV (2.10.0), nhiều máy gặp lỗi phiên bản khi `pip install nncase` kéo bản mới. Cách an toàn nhất:

1. Tải **best.onnx** về máy Windows
2. Chạy `convert_to_kmodel.py` trên máy (đã có trong project) — đúng bản nncase 2.10.0 + nncase_kpu 2.10.0 (win_amd64 whl)

Nếu bạn muốn convert trên Kaggle, cài đúng bản:
```
pip install nncase==2.10.0
curl -L -O https://github.com/kendryte/nncase/releases/download/v2.10.0/nncase_kpu-2.10.0-py2.py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
pip install that.whl
```